# 02 — Window-Specific Data Pulls

**Thesis:** Implied Volatility Smile Spillovers (AP-33)  
**Author:** Başar Hacımustafaoğlu — 1******6  

---

## Purpose

This notebook pulls ALL data for two specific time windows:

- **Window A — EU:** 2013-02-01 to 2013-03-31 (centred on EU sample data)
- **Window B — US:** 2014-02-01 to 2014-03-31 (centred on US sample data)

We pull a wider window than the option data itself (±1 month) to allow for lag construction in regressions.

### Why two windows?
The OptionMetrics sample data is locked:
- EU sample (`volatility_surface_2013`): March 2013 only
- US sample (`vsurfd2014`): March 2014 only
- These do NOT overlap — full OptionMetrics access is needed for the real thesis
- These two pipelines test the cleaning and merging logic before full access arrives

### What this notebook does NOT do
- No cleaning, no transformation, no analysis
- Only pulls and saves raw data for the two windows
- Cleaning happens in notebooks 03a (EU) and 03b (US)

---

> ⚠️ Run notebook 01 first to verify WRDS connection.

## Step 1 — Imports and connection

In [1]:
import os
import sys
import datetime
import json
import pandas as pd
import wrds
from dotenv import load_dotenv

sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
from wrds_utils import connect_wrds, validate_df

load_dotenv()
conn = connect_wrds()

TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# Output directories — separate folders for each window
RAW_EU  = "../data/raw/window_eu_2013"
RAW_US  = "../data/raw/window_us_2014"
LOG_DIR = "../logs"

os.makedirs(RAW_EU,  exist_ok=True)
os.makedirs(RAW_US,  exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# Window definitions
# Wider than the option data itself to allow lag construction
EU_START = '2013-01-01'
EU_END   = '2013-04-30'
US_START = '2014-01-01'
US_END   = '2014-04-30'

print(f"Run timestamp : {TIMESTAMP}")
print(f"EU window     : {EU_START} → {EU_END}")
print(f"US window     : {US_START} → {US_END}")

Connecting to WRDS as: basar
Loading library list...
Done
Connection established.
Run timestamp : 20260314_142434
EU window     : 2013-01-01 → 2013-04-30
US window     : 2014-01-01 → 2014-04-30


## Step 2 — Define a reusable pull function

This function pulls a table for a given date window, validates it, saves it, and logs the result.

In [2]:
pull_log = []

def pull_window(library, table, date_col, start, end, output_dir, label, extra_where=""):
    """
    Pull a table for a specific date window.
    
    Parameters
    ----------
    library     : WRDS library name
    table       : table name within the library
    date_col    : name of the date column to filter on
    start       : start date string 'YYYY-MM-DD'
    end         : end date string 'YYYY-MM-DD'
    output_dir  : folder to save the CSV
    label       : short label for the filename (e.g. 'EU2013' or 'US2014')
    extra_where : optional additional SQL WHERE clause (e.g. 'AND prccd IS NOT NULL')
    """
    key = f"{library}.{table}"
    print(f"\n{'='*60}")
    print(f"Pulling: {key} [{label}]")
    print(f"Window : {start} → {end}")
    print(f"{'='*60}")

    log_entry = {
        "library":   library,
        "table":     table,
        "label":     label,
        "start":     start,
        "end":       end,
        "status":    None,
        "n_rows":    None,
        "n_cols":    None,
        "columns":   None,
        "error":     None,
        "saved_to":  None,
    }

    try:
        sql = f"""
            SELECT * FROM {library}.{table}
            WHERE {date_col} BETWEEN '{start}' AND '{end}'
            {extra_where}
            ORDER BY {date_col}
        """
        df = conn.raw_sql(sql)

        log_entry["status"]  = "SUCCESS"
        log_entry["n_rows"]  = len(df)
        log_entry["n_cols"]  = len(df.columns)
        log_entry["columns"] = list(df.columns)

        validate_df(df, f"{key} [{label}]")

        filename = f"{library}__{table}__{label}__{TIMESTAMP}.csv"
        filepath = os.path.join(output_dir, filename)
        df.to_csv(filepath, index=False)
        log_entry["saved_to"] = filepath
        print(f"\n✓ {len(df)} rows saved to: {filepath}")

    except Exception as e:
        log_entry["status"] = "ERROR"
        log_entry["error"]  = str(e)
        print(f"\n✗ ERROR: {e}")

    pull_log.append(log_entry)
    return log_entry

print("Pull function defined.")

Pull function defined.


## Step 3 — Pull EU window (2013)

All tables pulled for the EU window: 2013-01-01 → 2013-04-30

In [3]:
# ----------------------------------------------------------------
# CORE: OptionMetrics Europe sample
# ----------------------------------------------------------------

# Volatility surface — PRIMARY thesis variable (EU side)
pull_window("optionmsamp_europe", "volatility_surface_2013",
            "date", EU_START, EU_END, RAW_EU, "EU2013")

# Raw option prices
pull_window("optionmsamp_europe", "option_price_2013",
            "date", EU_START, EU_END, RAW_EU, "EU2013")

# Underlying security prices
pull_window("optionmsamp_europe", "security_price",
            "date", EU_START, EU_END, RAW_EU, "EU2013")

# Historical (realized) volatility
pull_window("optionmsamp_europe", "historical_volatility",
            "date", EU_START, EU_END, RAW_EU, "EU2013")

# Security names — no date filter needed, pull full table
print(f"\n{'='*60}")
print("Pulling: optionmsamp_europe.security_name [EU2013] (full table)")
print(f"{'='*60}")
df_eu_names = conn.raw_sql("SELECT * FROM optionmsamp_europe.security_name")
df_eu_names.to_csv(os.path.join(RAW_EU, f"optionmsamp_europe__security_name__EU2013__{TIMESTAMP}.csv"), index=False)
print(f"✓ {len(df_eu_names)} rows saved")
print(f"Columns: {list(df_eu_names.columns)}")
print(df_eu_names.to_string())


Pulling: optionmsamp_europe.volatility_surface_2013 [EU2013]
Window : 2013-01-01 → 2013-04-30

VALIDATION REPORT: optionmsamp_europe.volatility_surface_2013 [EU2013]

[1] Shape: 2,860 rows x 10 columns

[2] Columns and dtypes:
    securityid                          Float64
    days                                Float64
    delta                               Int64
    callput                             string
    impliedvol                          Float64
    strike                              Float64
    premium                             Float64
    dispersion                          Float64
    currency                            Float64
    date                                string

[3] Missingness:
    No missing values detected.

[4] Date coverage:
    date: 2013-03-01 00:00:00 → 2013-03-15 00:00:00

[5] Duplicate rows: 0



✓ 2860 rows saved to: ../data/raw/window_eu_2013/optionmsamp_europe__volatility_surface_2013__EU2013__20260314_142434.csv

Pulling: optionmsamp_euro

In [4]:
# ----------------------------------------------------------------
# CONTROLS: aligned to EU window (2013)
# ----------------------------------------------------------------

# US risk-free rates (FRB) — used for Black-Scholes inputs
pull_window("frb", "rates_daily",
            "date", EU_START, EU_END, RAW_EU, "EU2013")

# CRSP daily market index — US market benchmark
pull_window("crsp", "dsi",
            "date", EU_START, EU_END, RAW_EU, "EU2013")

# CRSP S&P 500 index
pull_window("crsp", "dsp500",
            "caldt", EU_START, EU_END, RAW_EU, "EU2013")

# Fama-French daily factors
pull_window("ff", "factors_daily",
            "date", EU_START, EU_END, RAW_EU, "EU2013")

# CBOE VIX
pull_window("cboe", "cboe",
            "date", EU_START, EU_END, RAW_EU, "EU2013")

# Compustat Global daily index prices
pull_window("comp", "g_idx_daily",
            "datadate", EU_START, EU_END, RAW_EU, "EU2013")

# Compustat Global exchange rates (EUR/USD)
pull_window("comp", "g_exrt_dly",
            "datadate", EU_START, EU_END, RAW_EU, "EU2013")

# EU short selling data
pull_window("wrdsapps", "eushort",
            "position_date", EU_START, EU_END, RAW_EU, "EU2013")

print("\n✓ EU window pulls complete.")


Pulling: frb.rates_daily [EU2013]
Window : 2013-01-01 → 2013-04-30

VALIDATION REPORT: frb.rates_daily [EU2013]

[1] Shape: 120 rows x 83 columns

[2] Columns and dtypes:
    date                                string
    daaa                                Float64
    dbaa                                Float64
    dcd1m                               Float64
    dcd90                               Float64
    dcd6m                               Float64
    h1rifsgfpam03nb                     string
    h0rifsgfpam06nb                     string
    h0rifsgfpay01nb                     string
    dbkac                               string
    d_ba_m6                             string
    dltboard                            string
    d_cp_m1                             string
    d_cp_m3                             string
    d_cp_m6                             string
    rifsppcusnb                         string
    rifsppcunb                          string
    d_dwb_na            

## Step 4 — Pull US window (2014)

All tables pulled for the US window: 2014-01-01 → 2014-04-30

In [5]:
# ----------------------------------------------------------------
# CORE: OptionMetrics US sample
# ----------------------------------------------------------------

# Volatility surface — PRIMARY thesis variable (US side)
pull_window("optionmsamp_us", "vsurfd2014",
            "date", US_START, US_END, RAW_US, "US2014")

# Raw option prices
pull_window("optionmsamp_us", "opprcd2014",
            "date", US_START, US_END, RAW_US, "US2014")

# Underlying security prices
pull_window("optionmsamp_us", "secprd",
            "date", US_START, US_END, RAW_US, "US2014")

# Security names — no date filter needed, pull full table
print(f"\n{'='*60}")
print("Pulling: optionmsamp_us.secnmd [US2014] (full table)")
print(f"{'='*60}")
df_us_names = conn.raw_sql("SELECT * FROM optionmsamp_us.secnmd")
df_us_names.to_csv(os.path.join(RAW_US, f"optionmsamp_us__secnmd__US2014__{TIMESTAMP}.csv"), index=False)
print(f"✓ {len(df_us_names)} rows saved")
print(f"Columns: {list(df_us_names.columns)}")
print(df_us_names.to_string())


Pulling: optionmsamp_us.vsurfd2014 [US2014]
Window : 2014-01-01 → 2014-04-30

VALIDATION REPORT: optionmsamp_us.vsurfd2014 [US2014]

[1] Shape: 2,600 rows x 9 columns

[2] Columns and dtypes:
    secid                               Float64
    date                                string
    days                                Float64
    delta                               Float64
    impl_volatility                     Float64
    impl_strike                         Float64
    impl_premium                        Float64
    dispersion                          Float64
    cp_flag                             string

[3] Missingness:
    No missing values detected.

[4] Date coverage:
    date: 2014-03-03 00:00:00 → 2014-03-14 00:00:00

[5] Duplicate rows: 0



✓ 2600 rows saved to: ../data/raw/window_us_2014/optionmsamp_us__vsurfd2014__US2014__20260314_142434.csv

Pulling: optionmsamp_us.opprcd2014 [US2014]
Window : 2014-01-01 → 2014-04-30

VALIDATION REPORT: optionmsamp_us.opprcd2014 

In [6]:
# ----------------------------------------------------------------
# CONTROLS: aligned to US window (2014)
# ----------------------------------------------------------------

# US risk-free rates (FRB)
pull_window("frb", "rates_daily",
            "date", US_START, US_END, RAW_US, "US2014")

# CRSP daily market index
pull_window("crsp", "dsi",
            "date", US_START, US_END, RAW_US, "US2014")

# CRSP S&P 500 index
pull_window("crsp", "dsp500",
            "caldt", US_START, US_END, RAW_US, "US2014")

# Fama-French daily factors
pull_window("ff", "factors_daily",
            "date", US_START, US_END, RAW_US, "US2014")

# CBOE VIX
pull_window("cboe", "cboe",
            "date", US_START, US_END, RAW_US, "US2014")

# Compustat Global daily index prices
pull_window("comp", "g_idx_daily",
            "datadate", US_START, US_END, RAW_US, "US2014")

# Compustat Global exchange rates
pull_window("comp", "g_exrt_dly",
            "datadate", US_START, US_END, RAW_US, "US2014")

# EU short selling (for cross-market signal)
pull_window("wrdsapps", "eushort",
            "position_date", US_START, US_END, RAW_US, "US2014")

print("\n✓ US window pulls complete.")


Pulling: frb.rates_daily [US2014]
Window : 2014-01-01 → 2014-04-30

VALIDATION REPORT: frb.rates_daily [US2014]

[1] Shape: 120 rows x 83 columns

[2] Columns and dtypes:
    date                                string
    daaa                                Float64
    dbaa                                Float64
    dcd1m                               string
    dcd90                               string
    dcd6m                               string
    h1rifsgfpam03nb                     string
    h0rifsgfpam06nb                     string
    h0rifsgfpay01nb                     string
    dbkac                               string
    d_ba_m6                             string
    dltboard                            string
    d_cp_m1                             string
    d_cp_m3                             string
    d_cp_m6                             string
    rifsppcusnb                         string
    rifsppcunb                          string
    d_dwb_na               

## Step 5 — Pull summary

In [7]:
summary_df = pd.DataFrame(pull_log)

print("PULL SUMMARY")
print("=" * 80)
print(summary_df[["library", "table", "label", "status", "n_rows", "n_cols"]].to_string(index=False))

n_success = (summary_df["status"] == "SUCCESS").sum()
n_error   = (summary_df["status"] == "ERROR").sum()
print(f"\nSuccessful : {n_success}")
print(f"Failed     : {n_error}")

if n_error > 0:
    print("\nFailed tables:")
    for _, row in summary_df[summary_df["status"] == "ERROR"].iterrows():
        print(f"  {row['library']}.{row['table']} [{row['label']}]: {row['error']}")

PULL SUMMARY
           library                   table  label  status  n_rows  n_cols
optionmsamp_europe volatility_surface_2013 EU2013 SUCCESS    2860      10
optionmsamp_europe       option_price_2013 EU2013 SUCCESS    8740      24
optionmsamp_europe          security_price EU2013 SUCCESS      50      14
optionmsamp_europe   historical_volatility EU2013 SUCCESS     143       5
               frb             rates_daily EU2013 SUCCESS     120      83
              crsp                     dsi EU2013 SUCCESS      82      11
              crsp                  dsp500 EU2013 SUCCESS      82      11
                ff           factors_daily EU2013 SUCCESS      82       6
              cboe                    cboe EU2013 SUCCESS      82      17
              comp             g_idx_daily EU2013 SUCCESS   45307      10
              comp              g_exrt_dly EU2013 SUCCESS   20719       5
          wrdsapps                 eushort EU2013 SUCCESS    6695      10
    optionmsamp_us       

## Step 6 — Verify output files

In [8]:
print("EU window files:")
eu_files = sorted(os.listdir(RAW_EU))
for f in eu_files:
    filepath = os.path.join(RAW_EU, f)
    df = pd.read_csv(filepath)
    print(f"  {f}")
    print(f"    → {df.shape[0]} rows x {df.shape[1]} cols")

print(f"\nUS window files:")
us_files = sorted(os.listdir(RAW_US))
for f in us_files:
    filepath = os.path.join(RAW_US, f)
    df = pd.read_csv(filepath)
    print(f"  {f}")
    print(f"    → {df.shape[0]} rows x {df.shape[1]} cols")

EU window files:
  cboe__cboe__EU2013__20260314_142434.csv
    → 82 rows x 17 cols
  comp__g_exrt_dly__EU2013__20260314_142434.csv
    → 20719 rows x 5 cols
  comp__g_idx_daily__EU2013__20260314_142434.csv
    → 45307 rows x 10 cols
  crsp__dsi__EU2013__20260314_142434.csv
    → 82 rows x 11 cols
  crsp__dsp500__EU2013__20260314_142434.csv
    → 82 rows x 11 cols
  ff__factors_daily__EU2013__20260314_142434.csv
    → 82 rows x 6 cols
  frb__rates_daily__EU2013__20260314_142434.csv
    → 120 rows x 83 cols
  optionmsamp_europe__historical_volatility__EU2013__20260314_142434.csv
    → 143 rows x 5 cols
  optionmsamp_europe__option_price_2013__EU2013__20260314_142434.csv
    → 8740 rows x 24 cols
  optionmsamp_europe__security_name__EU2013__20260314_142434.csv
    → 1 rows x 6 cols
  optionmsamp_europe__security_price__EU2013__20260314_142434.csv
    → 50 rows x 14 cols
  optionmsamp_europe__volatility_surface_2013__EU2013__20260314_142434.csv
    → 2860 rows x 10 cols
  wrdsapps__eushort

## Step 7 — Save pull log

In [9]:
log_path = os.path.join(LOG_DIR, f"window_pulls_log_{TIMESTAMP}.json")

with open(log_path, "w") as f:
    json.dump(pull_log, f, indent=2, default=str)

print(f"Pull log saved to: {log_path}")

Pull log saved to: ../logs/window_pulls_log_20260314_142434.json


## Step 8 — Close connection

In [10]:
try:
    conn.close()
    print("WRDS connection closed.")
except Exception as e:
    print(f"Connection already closed or not active: {e}")

print(f"\nNotebook 02 complete.")
print("\nNext steps:")
print("  - Check data/raw/window_eu_2013/ for EU window CSVs")
print("  - Check data/raw/window_us_2014/ for US window CSVs")
print("  - Proceed to notebook 03a (EU pipeline) and 03b (US pipeline)")

WRDS connection closed.

Notebook 02 complete.

Next steps:
  - Check data/raw/window_eu_2013/ for EU window CSVs
  - Check data/raw/window_us_2014/ for US window CSVs
  - Proceed to notebook 03a (EU pipeline) and 03b (US pipeline)
